# Public Transit Honor System vs. Fare Gates

Michael Tian, Michael Mehall, Evan Crow, Leandros Manwaring, Ian Solberg 

April 2026

---

In [20]:
import pandas as pd
import numpy as np
from Research_Framework.ResearchHandler import ResearchHandler
import Research_Framework.transforms as tf
from mbta_handling import blank, build_policy_flow 

---

#### Setup Cell, do not execute - files too large for GH storage.

In [21]:
df = build_policy_flow(nodes_fp="data/mbta_rapid_transit/MBTA_NODE.shp", gse_fp="data/GSE/GSE.csv", travel_times_fp="data/TravelTimes_2026/2026-02_HRTravelTimes.csv")
mbta = ResearchHandler(df, handling_function=blank, initialize=False)
# mbta.data.to_csv("data/output_data/mbta_gate_flows_clean.csv", index=False)

/Users/ian/Desktop/MAINFRAME/School/Year 3/Year3Sem2/Behavioral Economics/MBTA-Fare-Analysis/mbta_handling.py:40: DtypeWarning: Columns (0: trip_id, 1: to_stop_id) have mixed types. Specify dtype option on import or set low_memory=False.
  travel_times = pd.read_csv(travel_times_fp)


Issue initializing data. ('STATION', 'LINE')


---

## Real Setup:

In [29]:
fp = "data/output_data/mbta_gate_flows_clean.csv"
mbta = ResearchHandler(fp, handling_function=blank, initialize=True)
mbta.data

,station_line_source,station_line_destination,line,time_bucket,avg_travel_time_sec,source_avg_gated_entries,dest_avg_gated_entries
0,"Kendall/MIT, RED","Wollaston, RED",RED,PM Rush,1527.120163,3996.478442,212.271115
1,"Kendall/MIT, RED","Park Street, RED",RED,PM Rush,227.785782,3996.478442,1481.583148
2,"Kendall/MIT, RED","Savin Hill, RED",RED,PM Rush,1072.838900,3996.478442,238.299365
3,"Kendall/MIT, RED","Braintree, RED",RED,PM Rush,2253.193416,3996.478442,359.482714
4,"Kendall/MIT, RED","Quincy Center, RED",RED,PM Rush,1712.395538,3996.478442,525.383793
...,...,...,...,...,...,...,...
5553,"State, ORANGE","Wellington, ORANGE",ORANGE,Early Morning,642.332418,0.000000,421.001378
5554,"State, ORANGE","Wellington, ORANGE",ORANGE,Evening,635.147175,0.000000,183.455115
5555,"State, ORANGE","Wellington, ORANGE",ORANGE,Late Night,592.536170,0.000000,85.692929
5556,"State, ORANGE","Wellington, ORANGE",ORANGE,Midday,684.000000,0.000000,843.853244


---

# MBTA Policy Flow Dataset

**5,558 rows** — each row is a directed station pair on a rapid transit line during a time-of-day bucket, with average travel time and fare gate activity for February 2026.

## Columns

| Column | Description |
|--------|-------------|
| `station_line_source` | Origin station and line (e.g. `Kendall/MIT, RED`) |
| `station_line_destination` | Destination station and line |
| `line` | Rapid transit line: RED, ORANGE, or BLUE |
| `time_bucket` | Period of day: Early Morning, AM Rush, Midday, PM Rush, Evening, Late Night |
| `avg_travel_time_sec` | Mean travel time in seconds across all February 2026 trips |
| `source_avg_gated_entries` | Mean fare gate entries at origin station during that time bucket |
| `dest_avg_gated_entries` | Mean fare gate entries at destination station during that time bucket |

## Time Buckets

| Bucket | Hours |
|--------|-------|
| Early Morning | 5:00–7:00 |
| AM Rush | 7:00–10:00 |
| Midday | 10:00–15:00 |
| PM Rush | 15:00–19:00 |
| Evening | 19:00–22:00 |
| Late Night | 22:00–5:00 |

## Notes

Full Description of datasources and transformation can be found in `data/README.md`

---

# Analysis

In [30]:
# Extract station names
mbta.data["source_station"] = mbta.data["station_line_source"].str.rsplit(", ", n=1).str[0]
mbta.data["dest_station"] = mbta.data["station_line_destination"].str.rsplit(", ", n=1).str[0]

# Associated gate total
mbta.data["associated_gate_total"] = mbta.data["source_avg_gated_entries"] + mbta.data["dest_avg_gated_entries"]

# Short haul indicator (Barabino et al., 2015 — OR=0.7065, p=0.008)
mbta.data["short_haul"] = (mbta.data["avg_travel_time_sec"] < 900).astype(int)

# Short haul exposure per station
short_haul_exposure = (mbta.data
    .groupby("source_station")["short_haul"]
    .mean()
)
mbta.attach("short_haul_exposure",
    mbta.data["source_station"].map(short_haul_exposure))

# Number of connections per station (network centrality proxy)
n_connections = (mbta.data
    .groupby("source_station")["station_line_destination"]
    .nunique()
)
mbta.attach("n_connections",
    mbta.data["source_station"].map(n_connections))

# Average outbound travel time per station + time bucket
avg_outbound = (mbta.data
    .groupby(["source_station", "time_bucket"])["avg_travel_time_sec"]
    .mean()
)
mbta.attach("avg_outbound_travel_sec",
    mbta.data.set_index(["source_station", "time_bucket"]).index.map(avg_outbound))

# Normalized versions using your transforms
mbta.normalize_and_attach("associated_gate_total", tf.min_max_scale, "gate_total_scaled")
mbta.normalize_and_attach("short_haul_exposure", tf.min_max_scale, "short_haul_scaled")
mbta.normalize_and_attach("n_connections", tf.min_max_scale, "n_connections_scaled")

# Dummies
line_dummies = pd.get_dummies(mbta.data["line"], prefix="line", drop_first=True)
bucket_dummies = pd.get_dummies(mbta.data["time_bucket"], prefix="bucket", drop_first=True)

for col in line_dummies.columns:
    mbta.attach(col, line_dummies[col], quiet=True)

for col in bucket_dummies.columns:
    mbta.attach(col, bucket_dummies[col], quiet=True)

# Set up regression
mbta.clear_caches()
mbta.set_dependent("gate_total_scaled")
mbta.add_independents("short_haul_scaled")
mbta.add_controls(*line_dummies.columns, *bucket_dummies.columns)


Attached 'short_haul_exposure' to dataset
Attached 'n_connections' to dataset
Attached 'avg_outbound_travel_sec' to dataset
Created gate_total_scaled from associated_gate_total using function: min_max_scale and attached to dataset
Created short_haul_scaled from short_haul_exposure using function: min_max_scale and attached to dataset
Created n_connections_scaled from n_connections using function: min_max_scale and attached to dataset
Caches cleared
Dependent variable set to: gate_total_scaled
Independent variables: ['short_haul_scaled']
Control variables: ['line_ORANGE', 'line_RED', 'bucket_Early Morning', 'bucket_Evening', 'bucket_Late Night', 'bucket_Midday', 'bucket_PM Rush']


## Validate Data Shape

In [31]:
print("Verifying data integrity and shape...")
print(f"Data shape: {mbta.data.shape}")
print(f"Dependent variable missing values: {mbta.data['gate_total_scaled'].isnull().sum()}")
print(f"Independent variables missing values:")
print(f"  short_haul_scaled: {mbta.data['short_haul_scaled'].isnull().sum()}")
print(f"  n_connections_scaled: {mbta.data['n_connections_scaled'].isnull().sum()}")
print(f"  avg_outbound_travel_sec: {mbta.data['avg_outbound_travel_sec'].isnull().sum()}")
print(f"Control variables missing values:")
for col in line_dummies.columns:   
    print(f"  {col}: {mbta.data[col].isnull().sum()}")
for col in bucket_dummies.columns:
    print(f"  {col}: {mbta.data[col].isnull().sum()}")
assert len(mbta.dependent) == mbta.get_X().shape[0], "Dependent variable length does not match number of rows in X"
assert mbta.get_X().shape[1] == len(mbta.independents) + len(mbta.controls), "Number of columns in X does not match number of independents and controls"
print("Data integrity checks passed. Ready for regression analysis.")

Verifying data integrity and shape...
Data shape: (5558, 24)
Dependent variable missing values: 0
Independent variables missing values:
  short_haul_scaled: 0
  n_connections_scaled: 0
  avg_outbound_travel_sec: 0
Control variables missing values:
  line_ORANGE: 0
  line_RED: 0
  bucket_Early Morning: 0
  bucket_Evening: 0
  bucket_Late Night: 0
  bucket_Midday: 0
  bucket_PM Rush: 0
Data integrity checks passed. Ready for regression analysis.


## Models

#### Baseline OLS

In [32]:
X = mbta.get_X().astype(float)
y = mbta.get_y().astype(float)

X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      gate_total_scaled   R-squared:                       0.536
Model:                            OLS   Adj. R-squared:                  0.535
Method:                 Least Squares   F-statistic:                     800.3
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:48:04   Log-Likelihood:                 5074.7
No. Observations:                5558   AIC:                        -1.013e+04
Df Residuals:                    5549   BIC:                        -1.007e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.1350 

In [33]:
# Composite score: gate volume * evasion risk multiplier
mbta.apply_and_attach(
    ["source_avg_gated_entries", "short_haul_exposure"],
    lambda df: df["source_avg_gated_entries"] * (1 + df["short_haul_exposure"]),
    "enforcement_score_v2"
)

# Aggregate to station x time_bucket — use FIRST not mean (gate entries are same for all pairs from a station)
station_alloc = (mbta.data
    .groupby(["source_station", "time_bucket"])["enforcement_score_v2"]
    .first()
    .reset_index()
)

# % allocation within each time bucket
station_alloc["pct_allocation"] = (
    station_alloc.groupby("time_bucket")["enforcement_score_v2"]
    .transform(lambda x: x / x.sum() * 100)
)

allocation_table = (station_alloc
    .pivot(index="source_station", columns="time_bucket", values="pct_allocation")
    .fillna(0)
    .round(2)
)[["Early Morning", "AM Rush", "Midday", "PM Rush", "Evening", "Late Night"]]

allocation_table["daily_avg"] = allocation_table.mean(axis=1).round(2)
allocation_table = allocation_table.sort_values("daily_avg", ascending=False)

Created enforcement_score_v2 from ['source_avg_gated_entries', 'short_haul_exposure'] and attached to dataset


In [34]:
# Naive allocation: just gate volume, no evasion weighting
mbta.attach("naive_score", mbta.data["source_avg_gated_entries"])

naive_alloc = (mbta.data
    .groupby(["source_station", "time_bucket"])["naive_score"]
    .first()
    .reset_index()
)
naive_alloc["pct_naive"] = (
    naive_alloc.groupby("time_bucket")["naive_score"]
    .transform(lambda x: x / x.sum() * 100)
)

# Data-driven allocation (already built)
informed_alloc = (mbta.data
    .groupby(["source_station", "time_bucket"])["enforcement_score_v2"]
    .first()
    .reset_index()
)
informed_alloc["pct_informed"] = (
    informed_alloc.groupby("time_bucket")["enforcement_score_v2"]
    .transform(lambda x: x / x.sum() * 100)
)

# Merge and compute difference
comparison = naive_alloc[["source_station", "time_bucket", "pct_naive"]].merge(
    informed_alloc[["source_station", "time_bucket", "pct_informed"]],
    on=["source_station", "time_bucket"]
)
comparison["delta"] = comparison["pct_informed"] - comparison["pct_naive"]

# Pivot the delta for readability
delta_table = (comparison
    .pivot(index="source_station", columns="time_bucket", values="delta")
    .fillna(0)
    .round(2)
)[["Early Morning", "AM Rush", "Midday", "PM Rush", "Evening", "Late Night"]]

delta_table["avg_delta"] = delta_table.mean(axis=1).round(2)
delta_table = delta_table.sort_values("avg_delta", ascending=False)

# Also build summary tables for each approach
naive_table = (naive_alloc
    .pivot(index="source_station", columns="time_bucket", values="pct_naive")
    .fillna(0).round(2)
)[["Early Morning", "AM Rush", "Midday", "PM Rush", "Evening", "Late Night"]]
naive_table["daily_avg"] = naive_table.mean(axis=1).round(2)
naive_table = naive_table.sort_values("daily_avg", ascending=False)

delta_table.columns.name = None
naive_table.columns.name = None
allocation_table.columns.name = None

Attached 'naive_score' to dataset


In [28]:
naive_table.to_csv("data/output_data/naive_allocation.csv")
allocation_table.to_csv("data/output_data/informed_allocation.csv")
delta_table.to_csv("data/output_data/allocation_delta.csv")